# Mercado 1D + Greeks — NDX European Call

Notebook simples para:

1. carregar os dados reais do projeto (`get_data` / `adjust_data`);
2. escolher o contrato com mais observações;
3. calcular Black–Scholes e Greeks analíticas nos pontos reais;
4. plotar preço e Greeks em 1D;
5. carregar Greeks já calculadas pelo projeto (`experimentos_pinn/greeks/**/greeks_comparison.csv`);
6. fazer cortes 1D `S -> V, Delta, Gamma, Theta` para comparar modelos com Black–Scholes.

Sem retreinar modelo. Sem tempo de cálculo. Sem superfície 3D.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

PROJECT_ROOT = Path('.').resolve()
sys.path.append(str(PROJECT_ROOT))

from utils.get_data import get_data, adjust_data

RESULTS_DIR = PROJECT_ROOT / 'experimentos_pinn'
OUT_DIR = PROJECT_ROOT / 'figures_mercado_greeks_1d'
TAB_DIR = PROJECT_ROOT / 'tables_mercado_greeks_1d'
OUT_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)

MARKET = 'NDX'
R = 0.05
TAU_CUT = 0.50

plt.rcParams.update({
    'figure.figsize': (7, 4),
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.family': 'serif',
    'mathtext.fontset': 'stix',
    'axes.grid': True,
    'grid.color': '0.85',
    'grid.linewidth': 0.6,
    'axes.edgecolor': 'black',
    'axes.linewidth': 0.9,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.frameon': False,
})

COLORS = {
    'Market': 'black',
    'Black-Scholes': '#5E81B5',
    'CLASSICO': '#8FB131',
    'QNN': '#E19C24',
    'CQNN': '#80699B',
    'HQNN': '#C44E52',
}

def savefig(name):
    path = OUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    plt.show()
    print('salvo:', path)


## 1. Funções Black–Scholes e métricas

In [ ]:
def bs_call_greeks(S, tau, K, r, sigma):
    S = np.asarray(S, dtype=float)
    tau = np.asarray(tau, dtype=float)
    K = np.asarray(K, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    r = np.asarray(r, dtype=float)

    eps = 1e-10
    S_safe = np.maximum(S, eps)
    tau_safe = np.maximum(tau, eps)
    sigma_safe = np.maximum(sigma, eps)

    sqrt_tau = np.sqrt(tau_safe)
    d1 = (np.log(S_safe / K) + (r + 0.5 * sigma_safe**2) * tau_safe) / (sigma_safe * sqrt_tau)
    d2 = d1 - sigma_safe * sqrt_tau

    price = S_safe * norm.cdf(d1) - K * np.exp(-r * tau_safe) * norm.cdf(d2)
    delta = norm.cdf(d1)
    gamma = norm.pdf(d1) / (S_safe * sigma_safe * sqrt_tau)
    theta = -(S_safe * norm.pdf(d1) * sigma_safe) / (2 * sqrt_tau) - r * K * np.exp(-r * tau_safe) * norm.cdf(d2)

    payoff = np.maximum(S - K, 0.0)
    price = np.where(tau <= eps, payoff, price)
    delta = np.where(tau <= eps, (S > K).astype(float), delta)
    gamma = np.where((tau <= eps) | (S <= eps), np.nan, gamma)
    theta = np.where(tau <= eps, np.nan, theta)
    price = np.where(S <= eps, 0.0, price)

    return price, delta, gamma, theta


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    err = y_pred - y_true
    return pd.Series({
        'N': len(y_true),
        'RMSE': np.sqrt(np.mean(err**2)) if len(err) else np.nan,
        'MAE': np.mean(np.abs(err)) if len(err) else np.nan,
        'Bias': np.mean(err) if len(err) else np.nan,
        'Corr': np.corrcoef(y_true, y_pred)[0, 1] if len(err) > 1 else np.nan,
    })


## 2. Carregar mercado do projeto e escolher um contrato

In [ ]:
data_yf, opt = get_data(MARKET)

# O adjust_data() remove as linhas sem preço no Yahoo Finance.
# Por isso NÃO dá para copiar opt['date'] direto para market.
# Fazemos o merge aqui mantendo a coluna date já alinhada.
opt = opt.copy()
opt['date'] = pd.to_datetime(opt['date'])

px = data_yf.reset_index().copy()
px['Date'] = pd.to_datetime(px['Date'])

merged = opt.merge(px, left_on='date', right_on='Date', suffixes=('', f'_{MARKET}'))

market = merged[[
    'date',
    'Close',
    'current_time',
    'strike_price',
    'best_bid',
    'best_offer',
    'ticker',
    'impl_volatility',
]].copy()

market = market.rename(columns={'Close': 'spot_price'})

# Mesmo ajuste usado no projeto.
market['strike_price'] = market['strike_price'] / 1000
market['market_price'] = (market['best_bid'] + market['best_offer']) / 2
market['moneyness'] = market['spot_price'] / market['strike_price']
market['tau'] = market['current_time']
market['r'] = R

if market['impl_volatility'].median() > 3:
    market['impl_volatility'] = market['impl_volatility'] / 100

if market['tau'].median() > 5:
    market['tau'] = market['tau'] / 365

selected_ticker = market['ticker'].value_counts().idxmax()
df = market[market['ticker'] == selected_ticker].copy().sort_values('date').reset_index(drop=True)

print('ticker escolhido:', selected_ticker)
print('N:', len(df))
print('K médio:', df['strike_price'].mean())
print('spot min/max:', df['spot_price'].min(), df['spot_price'].max())
print('moneyness min/max:', df['moneyness'].min(), df['moneyness'].max())
print('tau min/max:', df['tau'].min(), df['tau'].max())

display(df.head())

df.to_csv(TAB_DIR / 'market_selected_raw.csv', index=False)


## 3. Black–Scholes e Greeks nos pontos reais de mercado

In [ ]:
df['bs_price'], df['bs_delta'], df['bs_gamma'], df['bs_theta'] = bs_call_greeks(
    S=df['spot_price'],
    tau=df['tau'],
    K=df['strike_price'],
    r=df['r'],
    sigma=df['impl_volatility'],
)

df['bs_error'] = df['bs_price'] - df['market_price']
df['bs_abs_error'] = df['bs_error'].abs()

df['moneyness_region'] = pd.cut(
    df['moneyness'],
    bins=[-np.inf, 0.97, 1.03, np.inf],
    labels=['OTM', 'ATM', 'ITM'],
)

df['tau_region'] = pd.qcut(
    df['tau'], q=3, labels=['Short', 'Medium', 'Long'], duplicates='drop'
)

df.to_csv(TAB_DIR / 'market_selected_with_bs_greeks.csv', index=False)

display(metrics(df['market_price'], df['bs_price']).to_frame('Black-Scholes'))


## 4. Plots 1D — mercado, preço e resíduos

In [ ]:
plt.figure()
plt.plot(df['date'], df['market_price'], 'ko-', label='Market midpoint', markersize=4)
plt.plot(df['date'], df['bs_price'], '-', color=COLORS['Black-Scholes'], label='Black-Scholes', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Option price')
plt.title('NDX European Call — market vs Black-Scholes')
plt.legend()
savefig('01_market_vs_bs_time.png')


df_spot = df.sort_values('spot_price')
plt.figure()
plt.plot(df_spot['spot_price'], df_spot['market_price'], 'ko', label='Market midpoint', markersize=4)
plt.plot(df_spot['spot_price'], df_spot['bs_price'], '-', color=COLORS['Black-Scholes'], label='Black-Scholes', linewidth=2)
plt.xlabel(r'Spot price $S$')
plt.ylabel('Option price')
plt.title('NDX European Call — 1D price curve')
plt.legend()
savefig('02_market_vs_bs_spot.png')


plt.figure()
plt.axhline(0, color='black', linewidth=1)
plt.plot(df['date'], df['bs_error'], 'o-', color=COLORS['Black-Scholes'], markersize=4)
plt.xlabel('Date')
plt.ylabel(r'$V_{BS}-V_{market}$')
plt.title('Black-Scholes residual over time')
savefig('03_bs_residual_time.png')


plt.figure(figsize=(4.8, 4.4))
plt.scatter(df['market_price'], df['bs_price'], s=35, edgecolor='black', linewidth=0.4, color=COLORS['Black-Scholes'])
lo = min(df['market_price'].min(), df['bs_price'].min())
hi = max(df['market_price'].max(), df['bs_price'].max())
plt.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
plt.xlabel('Market price')
plt.ylabel('Black-Scholes price')
plt.title('Market vs Black-Scholes')
savefig('04_market_vs_bs_scatter.png')


## 5. Plots 1D — Greeks Black–Scholes ao longo do mercado

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

axes[0].plot(df['date'], df['bs_delta'], 'o-', color='#5E81B5', markersize=4)
axes[0].set_ylabel(r'$\Delta$')
axes[0].set_title('Black-Scholes Greeks on market points')

axes[1].plot(df['date'], df['bs_gamma'], 'o-', color='#E19C24', markersize=4)
axes[1].set_ylabel(r'$\Gamma$')

axes[2].plot(df['date'], df['bs_theta'], 'o-', color='#C44E52', markersize=4)
axes[2].set_ylabel(r'$\Theta$')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.savefig(OUT_DIR / '05_bs_greeks_time.png', bbox_inches='tight')
plt.show()


fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

df_spot = df.sort_values('spot_price')
axes[0].plot(df_spot['spot_price'], df_spot['bs_delta'], 'o-', color='#5E81B5', markersize=4)
axes[0].set_ylabel(r'$\Delta$')
axes[0].set_title('Black-Scholes Greeks vs spot')

axes[1].plot(df_spot['spot_price'], df_spot['bs_gamma'], 'o-', color='#E19C24', markersize=4)
axes[1].set_ylabel(r'$\Gamma$')

axes[2].plot(df_spot['spot_price'], df_spot['bs_theta'], 'o-', color='#C44E52', markersize=4)
axes[2].set_ylabel(r'$\Theta$')
axes[2].set_xlabel(r'Spot price $S$')

plt.tight_layout()
plt.savefig(OUT_DIR / '06_bs_greeks_spot.png', bbox_inches='tight')
plt.show()


## 6. Métricas de mercado por região

In [ ]:
region_m = df.groupby('moneyness_region').apply(
    lambda g: metrics(g['market_price'], g['bs_price'])
).reset_index()

tau_m = df.groupby('tau_region').apply(
    lambda g: metrics(g['market_price'], g['bs_price'])
).reset_index()

region_m.to_csv(TAB_DIR / 'bs_metrics_by_moneyness.csv', index=False)
tau_m.to_csv(TAB_DIR / 'bs_metrics_by_tau.csv', index=False)

display(region_m)
display(tau_m)

plt.figure()
plt.bar(region_m['moneyness_region'].astype(str), region_m['RMSE'], color='#5E81B5')
plt.xlabel('Moneyness region')
plt.ylabel('RMSE vs market')
plt.title('Black-Scholes error by moneyness')
savefig('07_bs_rmse_by_moneyness.png')

plt.figure()
plt.bar(tau_m['tau_region'].astype(str), tau_m['RMSE'], color='#5E81B5')
plt.xlabel('Maturity region')
plt.ylabel('RMSE vs market')
plt.title('Black-Scholes error by maturity')
savefig('08_bs_rmse_by_maturity.png')


## 7. Carregar Greeks calculadas pelo projeto

Este bloco lê os arquivos gerados por `calculate_greeks_from_runs.py`:

```text
experimentos_pinn/greeks/<model_type>/<run_id>/greeks_comparison.csv
```

Ele não recalcula nada.

In [ ]:
greek_files = sorted((RESULTS_DIR / 'greeks').glob('*/*/greeks_comparison.csv'))
print('arquivos greeks_comparison encontrados:', len(greek_files))

parts = []
for path in greek_files:
    g = pd.read_csv(path)
    g['model_type'] = path.parents[1].name
    g['run_id'] = path.parents[0].name
    parts.append(g)

if parts:
    greeks = pd.concat(parts, ignore_index=True)
    greeks['family'] = greeks['model_type'].str.upper()
    greeks.to_csv(TAB_DIR / 'project_greeks_all.csv', index=False)
    display(greeks.head())
else:
    greeks = pd.DataFrame()
    print('Nenhuma Greeks salva encontrada. Rode calculate_greeks_from_runs.py se quiser comparar modelos.')


## 8. Resumo das Greeks por modelo/família

In [ ]:
if not greeks.empty:
    rows = []
    for (family, run_id), g in greeks.groupby(['family', 'run_id']):
        row = {'family': family, 'run_id': run_id, 'N': len(g)}
        for key in ['V', 'delta', 'gamma', 'theta']:
            pred = f'{key}_pred_un'
            true = f'{key}_true_un'
            if pred in g.columns and true in g.columns:
                m = metrics(g[true], g[pred])
                row[f'{key}_RMSE'] = m['RMSE']
                row[f'{key}_MAE'] = m['MAE']
                row[f'{key}_Bias'] = m['Bias']
        rows.append(row)

    greek_summary = pd.DataFrame(rows).sort_values(['V_RMSE', 'delta_RMSE'])
    greek_summary.to_csv(TAB_DIR / 'project_greeks_summary_simple.csv', index=False)
    display(greek_summary.head(20))
else:
    greek_summary = pd.DataFrame()


## 9. Cortes 1D das Greeks salvas: Black–Scholes vs modelos

Aqui fica a parte mais parecida com Trahan: corte 1D da solução aprendida e comparação com solução analítica.

In [ ]:
def choose_best_run_per_family(greek_summary, metric='V_RMSE'):
    if greek_summary.empty:
        return pd.DataFrame()
    return greek_summary.sort_values(metric).groupby('family', as_index=False).first()

best_greek_runs = choose_best_run_per_family(greek_summary, metric='V_RMSE')
display(best_greek_runs)


def plot_greek_cut(key, tau_cut=TAU_CUT):
    if greeks.empty or best_greek_runs.empty:
        print('Sem greeks salvas para plotar.')
        return

    plt.figure(figsize=(7.2, 4.2))

    for _, row in best_greek_runs.iterrows():
        fam = row['family']
        run = row['run_id']
        g = greeks[(greeks['family'] == fam) & (greeks['run_id'] == run)].copy()

        t_near = g['t'].iloc[(g['t'] - tau_cut).abs().argmin()]
        cut = g[np.isclose(g['t'], t_near)].copy().sort_values('S')

        true_col = f'{key}_true_un'
        pred_col = f'{key}_pred_un'

        if true_col not in cut.columns or pred_col not in cut.columns:
            continue

        if fam == best_greek_runs.iloc[0]['family']:
            plt.plot(cut['S'], cut[true_col], 'k-', linewidth=2.2, label='Black-Scholes')

        plt.plot(
            cut['S'], cut[pred_col], '--', linewidth=1.7,
            color=COLORS.get(fam, None), label=f'{fam}'
        )

    label = {'V': 'Price V', 'delta': r'$\Delta$', 'gamma': r'$\Gamma$', 'theta': r'$\Theta$'}[key]
    plt.xlabel(r'Spot price $S$')
    plt.ylabel(label)
    plt.title(f'1D cut near t/tau = {tau_cut:.2f} — {label}')
    plt.legend()
    savefig(f'09_cut_{key}_models_vs_bs.png')

for key in ['V', 'delta', 'gamma', 'theta']:
    plot_greek_cut(key, TAU_CUT)


## 10. Erro das Greeks por família

Esse bloco resume a comparação no espírito do Trahan: erro por família, sem tempo de cálculo.

In [ ]:
if not greek_summary.empty:
    family_greeks = greek_summary.groupby('family').agg({
        'V_RMSE': 'median',
        'delta_RMSE': 'median',
        'gamma_RMSE': 'median',
        'theta_RMSE': 'median',
    }).reset_index()

    family_greeks.to_csv(TAB_DIR / 'family_greeks_median_rmse.csv', index=False)
    display(family_greeks)

    for col, title in [
        ('V_RMSE', 'Price'),
        ('delta_RMSE', 'Delta'),
        ('gamma_RMSE', 'Gamma'),
        ('theta_RMSE', 'Theta'),
    ]:
        plt.figure()
        plt.bar(family_greeks['family'], family_greeks[col], color=[COLORS.get(f, '#777777') for f in family_greeks['family']])
        plt.yscale('log')
        plt.xlabel('Model family')
        plt.ylabel('Median RMSE')
        plt.title(f'{title} RMSE by family')
        savefig(f'10_{col}_by_family.png')
else:
    print('Sem resumo de greeks de modelos.')


## 11. Região difícil para Greeks: perto do strike

No caso de call, a região importante é perto de `S/K ≈ 1`, principalmente para Gamma. Este bloco calcula erros apenas nessa região para as Greeks salvas.

In [ ]:
if not greeks.empty:
    K_SYNTH = 40.0
    gg = greeks.copy()
    gg['moneyness'] = gg['S'] / K_SYNTH
    gg['region'] = pd.cut(
        gg['moneyness'],
        bins=[-np.inf, 0.97, 1.03, np.inf],
        labels=['OTM', 'ATM', 'ITM'],
    )

    rows = []
    for (family, run_id, region), g in gg.groupby(['family', 'run_id', 'region']):
        row = {'family': family, 'run_id': run_id, 'region': region, 'N': len(g)}
        for key in ['V', 'delta', 'gamma', 'theta']:
            pred = f'{key}_pred_un'
            true = f'{key}_true_un'
            if pred in g.columns and true in g.columns:
                row[f'{key}_RMSE'] = metrics(g[true], g[pred])['RMSE']
        rows.append(row)

    region_greeks = pd.DataFrame(rows)
    region_greeks.to_csv(TAB_DIR / 'project_greeks_by_moneyness_region.csv', index=False)
    display(region_greeks.head(30))

    atm = region_greeks[region_greeks['region'].astype(str) == 'ATM']
    if not atm.empty:
        atm_family = atm.groupby('family')[['V_RMSE', 'delta_RMSE', 'gamma_RMSE', 'theta_RMSE']].median().reset_index()
        display(atm_family)

        plt.figure()
        plt.bar(atm_family['family'], atm_family['gamma_RMSE'], color=[COLORS.get(f, '#777777') for f in atm_family['family']])
        plt.yscale('log')
        plt.xlabel('Model family')
        plt.ylabel(r'ATM $\Gamma$ RMSE')
        plt.title('Hard region: Gamma error near strike')
        savefig('11_atm_gamma_rmse_by_family.png')
else:
    print('Sem greeks salvas para análise regional.')


## 12. Arquivos gerados

As figuras ficam em `figures_mercado_greeks_1d/` e as tabelas em `tables_mercado_greeks_1d/`.
